# Unlocking Behavioral Intelligence in Airline Loyalty Programs
## Consulting & Analytics Club — IIT Guwahati | Summer Projects 2026

---

**Problem Statement:** Build a solution that helps the marketing team identify customers likely to disengage, understand which members are most valuable, and recommend actions to improve retention.

**Dataset:** ~16,700 Canadian loyalty members | Flight activity 2017–2018

**Deliverables:**
- Churn prediction model (XGBoost)
- Customer segmentation (K-Means)
- Retention strategy per segment
- Working Streamlit prototype

---

| Phase | Description | Output |
|---|---|---|
| Phase 1 | Data Understanding & Exploration | Cleaning log, key findings |
| Phase 2 | Feature Engineering & Churn Label | feature_matrix.csv |
| Phase 3 | ML Modelling — Churn + Segmentation | final_customer_segments.csv |


---
# Phase 1 — Data Understanding & Exploration

**Objective:** Load all dataset files, understand their structure, identify data quality issues, and confirm the files can be joined correctly. No data is changed in this phase — only observed and documented.

**Files loaded:**
- `Customer Loyalty History.csv` — 16,737 rows × 16 columns (one row per customer)
- `Customer Flight Activity.csv` — 392,936 rows × 8 columns (one row per customer per month)
- `Calendar.csv` — date dimension file

**Key rule:** Observe everything. Fix nothing. Document everything found.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

def section(title):
    print("\n" + "=" * 60)
    print(f"  {title}")
    print("=" * 60)

def subsection(title):
    print(f"\n--- {title} ---")

loyalty  = pd.read_csv("Customer Loyalty History (1).csv")
flights  = pd.read_csv("Customer Flight Activity (1).csv")
calendar = pd.read_csv("Calendar (1).csv")

print(f"Loyalty History  : {loyalty.shape[0]:,} rows × {loyalty.shape[1]} columns")
print(f"Flight Activity  : {flights.shape[0]:,} rows × {flights.shape[1]} columns")
print(f"Calendar         : {calendar.shape[0]:,} rows × {calendar.shape[1]} columns")


### Step 1 — Schema Overview

Understanding the structure of each file before any analysis.
Every column's data type, null count, and unique values are inspected.

**Why:** Pandas silently loads numeric columns as `object` if a stray character exists.
Checking dtypes early prevents wrong calculations later.


In [ ]:
section("STEP 1 — Schema overview")

for label, df in [("LOYALTY HISTORY", loyalty), ("FLIGHT ACTIVITY", flights)]:
    subsection(label)
    info = pd.DataFrame({
        "dtype"  : df.dtypes.astype(str),
        "nulls"  : df.isnull().sum(),
        "null_%" : (df.isnull().mean() * 100).round(2),
        "unique" : df.nunique(),
    })
    print(info.to_string())


### Step 2 — Null Analysis

Identifying which columns have missing values and quantifying them.

**Key distinction:**
- `Salary` NaN → College students have no income. NaN = 0, not missing.
- `Cancellation Year/Month` NaN → customer never cancelled. NaN = active. **Keep untouched.**


In [ ]:
section("STEP 2 — Missing value analysis")

for label, df in [("Loyalty History", loyalty), ("Flight Activity", flights)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    if len(nulls) == 0:
        print(f"\n  {label}: No missing values ✓")
    else:
        print(f"\n  {label} — columns with nulls:")
        for col, count in nulls.items():
            pct = count / len(df) * 100
            print(f"    {col:<40} {count:>6,} nulls  ({pct:.1f}%)")


### Step 3 — Cancellation Status

Counting how many members formally cancelled vs remained enrolled.

**Why this matters:** If formal cancellation rate is low, it alone is insufficient as a
churn definition. We need behavioral signals to capture silent disengagers.


In [ ]:
subsection("Cancellation status")
cancelled = loyalty["Cancellation Year"].notna().sum()
total     = len(loyalty)
print(f"  Cancelled members  : {cancelled:,} ({cancelled/total*100:.1f}%)")
print(f"  Active members     : {total - cancelled:,} ({(total-cancelled)/total*100:.1f}%)")

print("\n  Cancellations by year:")
yr_counts = loyalty["Cancellation Year"].value_counts().sort_index()
for yr, cnt in yr_counts.items():
    bar = "█" * int(cnt / yr_counts.max() * 30)
    print(f"    {int(yr)}  {bar}  {cnt:,}")


**Finding:** Only 12.3% of members formally cancelled.
87.7% never cancelled — but this does NOT mean they are all actively flying.
Formal cancellation alone is too narrow. Behavioral churn definition is needed.


### Step 4 — Negative Salary Check

Checking for impossible values in the Salary column.


In [ ]:
subsection("Salary anomaly check")
neg_count = (loyalty['Salary'] < 0).sum()
print(f"  Negative salary values: {neg_count}")

if neg_count > 0:
    print("\n  Education breakdown of negative salary rows:")
    print(loyalty[loyalty['Salary'] < 0]['Education'].value_counts())


**Finding:** 20 negative salary values found across mixed education levels.
Confirmed as data entry typos — sign error. Magnitude is correct.
**Decision:** Convert to positive (absolute value). Document in cleaning log.


### Step 5 — Zero Activity Check

Counting monthly records with zero flights to understand inactivity baseline.

**Why:** If most monthly records are zero, single-month inactivity is normal
and cannot be used as a churn signal. Need a longer window.


In [ ]:
subsection("Zero-activity check (months with no flights)")
zero_months = (flights["Total Flights"] == 0).sum()
total_rows  = len(flights)
print(f"  Months with 0 flights booked: {zero_months:,} ({zero_months/total_rows*100:.1f}% of all monthly records)")

# Customer level — how many flights per customer across full period
member_flights = flights.groupby('Loyalty Number')['Total Flights'].sum()
print(f"\n  Customer-level flight statistics:")
print(member_flights.describe().round(1))


**Finding:** 54.5% of monthly records show zero flights — single month inactivity is completely normal.

**However:** At customer level, the median customer booked **34 flights** across 24 months.
A customer with **zero total flights** is genuinely anomalous — 3+ standard deviations below median.
This distinction directly informs our behavioral churn threshold in Phase 2.


### Step 6 — Join Integrity Check

Verifying that all loyalty members have flight records and vice versa.

**Why:** A bad join is invisible until it corrupts your model.
Members in loyalty only with no flights are likely churned.
Flight records with no loyalty profile are unusable for modelling.


In [ ]:
section("STEP 6 — Join integrity check")

loyalty_ids = set(loyalty["Loyalty Number"].unique())
flight_ids  = set(flights["Loyalty Number"].unique())

in_both      = loyalty_ids & flight_ids
only_loyalty = loyalty_ids - flight_ids
only_flight  = flight_ids  - loyalty_ids

print(f"\n  Members in both files        : {len(in_both):,}")
print(f"  In loyalty only (no activity): {len(only_loyalty):,}  ← likely inactive / churned")
print(f"  In flight only (no profile)  : {len(only_flight):,}  ← data quality issue if > 0")


**Finding:** Perfect join integrity.
- 16,737 members exist in both files
- 0 members in loyalty only
- 0 orphaned flight records

Merge is completely clean. Inner join will produce no data loss.


### Step 7 — CLV Distribution

Examining Customer Lifetime Value distribution before modelling.


In [ ]:
subsection("CLV distribution")
clv_clean = loyalty["CLV"].dropna()
print(f"  Min    : {clv_clean.min():,.2f}")
print(f"  Median : {clv_clean.median():,.2f}")
print(f"  Mean   : {clv_clean.mean():,.2f}")
print(f"  Max    : {clv_clean.max():,.2f}")
print(f"  Negative CLV values: {(clv_clean < 0).sum()}")

plt.figure(figsize=(8, 4))
plt.hist(clv_clean, bins=40, color="#534AB7", alpha=0.75, edgecolor="white")
plt.axvline(clv_clean.median(), color="#E8593C", linestyle="--", linewidth=1.5,
            label=f"Median: {clv_clean.median():,.0f}")
plt.title("CLV Distribution", fontsize=12)
plt.xlabel("CLV")
plt.ylabel("Members")
plt.legend()
plt.tight_layout()
plt.show()


**Finding:** Median CLV = 5,780. Heavily right-skewed — small number of members have very high CLV.
No negative values confirmed. Log transformation needed before clustering to prevent outliers dominating.


### Step 8 — Demographic Analysis

Understanding the composition of the loyalty program membership.


In [ ]:
subsection("Loyalty Card / Tier distribution")
print(loyalty["Loyalty Card"].value_counts())

subsection("Education distribution")
print(loyalty["Education"].value_counts())

subsection("Marital Status distribution")
print(loyalty["Marital Status"].value_counts())

subsection("Gender distribution")
print(loyalty["Gender"].value_counts())

subsection("Province distribution")
print(loyalty["Province"].value_counts())


**Key findings:**
- Loyalty tiers: Star 45.6%, Nova 33.9%, Aurora 20.5% — healthy pyramid structure
- Education: Bachelor dominant (62.6%), College 25.3% — explains salary NaN pattern
- Gender: Almost perfectly balanced 50/50 — no gender bias
- Province: Ontario (32.3%), BC (26.3%), Quebec (19.7%) — 78.3% from three provinces
- All demographic columns are completely clean — zero nulls


### Step 9 — Merge Preview

Preview merge to confirm join works correctly before Phase 2.
**This is exploratory only — not saved as the final dataset.**


In [ ]:
section("STEP 9 — Merge preview (loyalty + flights)")

merged = flights.merge(loyalty, on="Loyalty Number", how="inner")
print(f"  Merged shape: {merged.shape[0]:,} rows × {merged.shape[1]} columns")
print(f"  (Each row = one member's activity in one month)")


### Phase 1 — Cleaning Log

All data quality issues found and decisions made.

| Column | Issue | Decision | Reason | Rows Affected |
|---|---|---|---|---|
| Salary | NaN values for College students | Fill with 0 in Phase 2 | Students have no income — 0 is factually correct | 4,238 |
| Salary | 20 negative values, mixed education | Convert to positive (abs value) | Confirmed typos — sign error, magnitude correct | 20 |
| Cancellation Year/Month | NaN values present | Keep untouched | NaN = never cancelled = active customer | 14,670 |

### Phase 1 — Key Questions Answered Before Phase 2

| Question | Answer |
|---|---|
| Which columns define churn? | Cancellation Year (formal) + Total Flights (behavioral) |
| Members with no flight activity? | 0 — all 16,737 members have flight records |
| Is CLV reliable? | Yes — no negatives, clean distribution |
| Train/test split? | Train 2012-2016, Test 2017-2018 (data is 2017-2018 only) |
| Behavioral churn threshold? | Zero flights ever — median is 34 flights, so 0 is genuinely anomalous |


---
# Phase 2 — Feature Engineering & Churn Label

**Objective:** Transform 392,936 monthly rows into one clean row per customer
with a churn label and meaningful behavioral features — ready for machine learning.

**Input:** Raw CSV files (cleaned)
**Output:** `modelling2.csv` — 16,737 rows × 33 features

**The transformation:**
```
392,936 monthly rows  →  16,737 customer rows
No churn label        →  churned = 0 or 1
Separate files        →  One master feature matrix
Raw demographics      →  Encoded + scaled
No behavioral signals →  Recency, frequency, trend, seasonal features
```


### Step 1 — Load Files & Remove Duplicates


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

cfa = pd.read_csv("Customer Flight Activity (1).csv")
clh = pd.read_csv("Customer Loyalty History (1).csv")
cal = pd.read_csv("Calendar (1).csv")

# Remove duplicates
cfa = cfa.drop_duplicates()
clh = clh.drop_duplicates()
cal = cal.drop_duplicates()

print(f"Flight Activity  : {cfa.shape}")
print(f"Loyalty History  : {clh.shape}")
print(f"Calendar         : {cal.shape}")


### Step 2 — Fix Negative Salaries

**Issue:** 20 negative salary values found across mixed education levels.
**Decision:** Confirmed as typos — convert to positive equivalent.
**Why not group median?** Since these are confirmed sign errors, the magnitude is correct.
Converting to absolute value is the most accurate fix.


In [ ]:
# Verify before fix
print(f"Negative salaries before fix: {(clh['Salary'] < 0).sum()}")
print("\nEducation breakdown of negative salary rows:")
print(clh[clh['Salary'] < 0]['Education'].value_counts())

# Fix — convert negatives to positive (confirmed typos)
clh['Salary'] = clh['Salary'].abs()

# Verify after fix
print(f"\nNegative salaries after fix: {(clh['Salary'] < 0).sum()}")


### Step 3 — Fix Salary Nulls for College Students

**Issue:** NaN salary for College students.
**Decision:** Fill with 0.
**Why:** College students have no income — 0 is factually correct, not an imputation.
Confirmed by cross-referencing with Education column.


In [ ]:
# Fill NaN salary with 0 for College students
clh.loc[clh['Education'] == 'College', 'Salary'] = clh.loc[
    clh['Education'] == 'College', 'Salary'].fillna(0)

print(f"Null salaries remaining  : {clh['Salary'].isna().sum()}")
print(f"College students (0 sal) : {((clh['Education'] == 'College') & (clh['Salary'] == 0)).sum()}")


### Step 4 — Build Master Dataset

Merging flight activity + loyalty history on Loyalty Number.
Adding Quarter and Season from Month column.

**Why inner join:** Only keep customers present in both files — confirmed clean in Phase 1.
**Calendar file not used:** Quarter and Season derived directly from Month — cleaner and equivalent.


In [ ]:
# Merge
master = cfa.merge(clh, on='Loyalty Number', how='inner')

# Derive Quarter and Season from Month
master['Quarter'] = master['Month'].apply(lambda m: f'Q{(m-1) // 3 + 1}')

season_map = {
    1: 'Winter', 2: 'Winter',  3: 'Spring',
    4: 'Spring', 5: 'Spring',  6: 'Summer',
    7: 'Summer', 8: 'Summer',  9: 'Fall',
    10: 'Fall',  11: 'Fall',   12: 'Winter'
}
master['Season'] = master['Month'].map(season_map)

print(f"Master shape: {master.shape}")
print(f"Unique members: {master['Loyalty Number'].nunique():,}")
print(f"Columns: {list(master.columns)}")


### Step 5 — Build Churn Label

Two competing churn definitions built and compared.
Churn label is built separately then merged into feature matrix — keeps design clean.

**Formal churn:** Customer has a recorded Cancellation Year.
**Behavioral churn:** Customer had zero total flights across entire 2017-2018 period.

**Why "zero flights ever" and not 6-month or 12-month threshold:**
- The median customer booked 34 flights across 24 months
- A customer with 0 total flights is genuinely anomalous
- 6-month threshold flagged 94.2% as churned — meaningless for 24-month dataset
- 12-month threshold flagged 45.5% — still too broad
- Zero flights ever is clean, unambiguous, and directly supported by data


In [ ]:
# Build churn label
churn_df = clh[['Loyalty Number', 'Cancellation Year']].copy()

# Formal churn
churn_df['formally_cancelled'] = churn_df['Cancellation Year'].notna().astype(int)

# Behavioral churn — zero flights across entire period
member_flights = cfa.groupby('Loyalty Number')['Total Flights'].sum()
churn_df['zero_flights_ever'] = churn_df['Loyalty Number'].map(
    member_flights).fillna(0) == 0
churn_df['zero_flights_ever'] = churn_df['zero_flights_ever'].astype(int)

# Final combined label
churn_df['churned'] = ((churn_df['formally_cancelled'] == 1) |
                       (churn_df['zero_flights_ever'] == 1)).astype(int)

# Keep only what's needed
churn_df = churn_df[['Loyalty Number', 'churned']]

print("Churn label summary:")
print(f"  Formally cancelled : {(churn_df['churned'] == 1).sum():,}")
print(f"  Churn rate         : {churn_df['churned'].mean()*100:.1f}%")
print(churn_df['churned'].value_counts())


**Churn label result:**
- Not churned: 14,051 (84.0%)
- Churned: 2,686 (16.0%)

16% churn rate is realistic and defensible for an airline loyalty program.
Class imbalance (84/16) will be handled by `scale_pos_weight` in XGBoost.


### Step 6 — Demographic Features

Features extracted directly from Customer Loyalty History.
One row per customer — no aggregation needed.

**Tenure_Months** is derived from Enrollment Year and Month — captures how long
a member has been in the program. Longer tenure = more loyalty habits built.


In [ ]:
# Membership tenure in months
clh['Tenure_Months'] = (
    (2018 - clh['Enrollment Year']) * 12 +
    (12 - clh['Enrollment Month'])
)

# Select demographic features
demographic = clh[[
    'Loyalty Number', 'Gender', 'Education', 'Salary',
    'Marital Status', 'Loyalty Card', 'CLV',
    'Enrollment Type', 'Province', 'Tenure_Months'
]].copy()

print(f"Demographic features shape: {demographic.shape}")
print(f"Null values: {demographic.isnull().sum().sum()}")


### Step 7 — Geographic Feature: Region

Province reduced to 3 regions for cleaner signal.
11 province categories → 3 region categories.

**Why Region over Province:**
- Province has 11 unique values — sparse for small groups (Yukon: 110 members)
- Region consolidates into meaningful geographic blocks
- One hot encoding of Region is cleaner than label encoding of Province


In [ ]:
# Map Province to Region
region_map = {
    'Ontario'              : 'East',
    'Quebec'               : 'East',
    'New Brunswick'        : 'East',
    'Nova Scotia'          : 'East',
    'Prince Edward Island' : 'East',
    'Newfoundland'         : 'East',
    'Alberta'              : 'West',
    'British Columbia'     : 'West',
    'Manitoba'             : 'West',
    'Saskatchewan'         : 'West',
    'Yukon'                : 'North'
}

demographic['Region'] = demographic['Province'].map(region_map)
print("Region distribution:")
print(demographic['Region'].value_counts())


### Step 8 — Behavioral Features

Aggregating 391,014 monthly flight activity rows down to one row per customer.

**Why aggregation:** ML models need one row per customer.
Each feature answers a specific business question about the customer.


In [ ]:
# Flight volume features
flight_features = cfa.groupby('Loyalty Number').agg(
    Total_Flights        = ('Total Flights', 'sum'),
    Avg_Flights_Month    = ('Total Flights', 'mean'),
    Max_Flights_Month    = ('Total Flights', 'max'),
    Total_Distance       = ('Distance', 'sum'),
    Avg_Distance_Month   = ('Distance', 'mean')
).reset_index()

# Points features
points_features = cfa.groupby('Loyalty Number').agg(
    Total_Points_Acc  = ('Points Accumulated', 'sum'),
    Total_Points_Red  = ('Points Redeemed', 'sum'),
    Total_Dollar_Red  = ('Dollar Cost Points Redeemed', 'sum')
).reset_index()

# Redemption rate — how actively a member uses earned points
# Low rate may signal disengagement OR saving up; high rate near exit signals churning
points_features['Redemption_Rate'] = (
    points_features['Total_Points_Red'] /
    points_features['Total_Points_Acc'].replace(0, 1)
)

print(f"Flight features  : {flight_features.shape}")
print(f"Points features  : {points_features.shape}")


### Step 9 — Recency Features

**Recency = how recently a customer was active.**
This is the single strongest churn predictor — confirmed by SHAP in Phase 3.

A customer who flew last month is fundamentally different from one silent for 18 months.


In [ ]:
# Sequential period index — 2017 Jan=1 to 2018 Dec=24
cfa['Period'] = (cfa['Year'] - 2017) * 12 + cfa['Month']

# Last active period per customer
last_active = cfa[cfa['Total Flights'] > 0].groupby(
    'Loyalty Number')['Period'].max().reset_index()
last_active.columns = ['Loyalty Number', 'Last_Active_Period']

max_period = cfa['Period'].max()
last_active['Months_Since_Last_Flight'] = max_period - last_active['Last_Active_Period']

# Members with zero flights get maximum recency
all_members = pd.DataFrame({'Loyalty Number': cfa['Loyalty Number'].unique()})
recency = all_members.merge(last_active, on='Loyalty Number', how='left')
recency['Months_Since_Last_Flight'] = recency['Months_Since_Last_Flight'].fillna(max_period)
recency = recency[['Loyalty Number', 'Months_Since_Last_Flight']]

# Zero activity months count
zero_months = cfa[cfa['Total Flights'] == 0].groupby(
    'Loyalty Number')['Month'].count().reset_index()
zero_months.columns = ['Loyalty Number', 'Zero_Activity_Months']

print(f"Max period (Dec 2018): {max_period}")
print(f"Recency shape: {recency.shape}")
print(recency['Months_Since_Last_Flight'].describe().round(2))


### Step 10 — Seasonal Features

Capturing which seasons each customer typically flies in.

**Why seasonal features matter:**
- Summer-only flyers are seasonal travelers — different retention timing needed
- Year-round flyers are likely business travelers — more stable loyalty
- Seasonal inactivity (Fall, Winter) proved to be a top-5 churn predictor in Phase 3


In [ ]:
# Flights per season per customer
season_flights = master.groupby(
    ['Loyalty Number', 'Season'])['Total Flights'].sum().unstack(fill_value=0)
season_flights.columns = [f'Flights_{s}' for s in season_flights.columns]
season_flights = season_flights.reset_index()

print(f"Seasonal features shape: {season_flights.shape}")
print(season_flights.head(3))


### Step 11 — Combine into Feature Matrix

All feature groups merged into one clean table.
One row per customer with all features and churn label attached.


In [ ]:
# Start with demographics
feature_matrix = demographic.copy()

# Add churn label
feature_matrix = feature_matrix.merge(churn_df, on='Loyalty Number', how='left')

# Add behavioral features
feature_matrix = feature_matrix.merge(flight_features, on='Loyalty Number', how='left')
feature_matrix = feature_matrix.merge(points_features, on='Loyalty Number', how='left')
feature_matrix = feature_matrix.merge(recency, on='Loyalty Number', how='left')
feature_matrix = feature_matrix.merge(zero_months, on='Loyalty Number', how='left')
feature_matrix = feature_matrix.merge(season_flights, on='Loyalty Number', how='left')

# Fill nulls with 0
feature_matrix = feature_matrix.fillna(0)

print(f"Feature matrix shape : {feature_matrix.shape}")
print(f"Null values          : {feature_matrix.isnull().sum().sum()}")
print(f"Columns: {list(feature_matrix.columns)}")


### Step 12 — Feature Engineering: New Features

Three additional features derived from existing ones to capture richer signals.


In [ ]:
# Log transform CLV — right-skewed distribution, compress the long tail
import numpy as np
feature_matrix['CLV_log'] = np.log1p(feature_matrix['CLV'])

# Flight consistency score — regularity of flying vs silent periods
# High score = consistent flyer; Low score = bursty or inactive
feature_matrix['Flight_consistency_score'] = (
    feature_matrix['Total_Flights'] /
    (feature_matrix['Zero_Activity_Months'] + 1)
)

# Season concentration score — how concentrated flying is in one season
# Score close to 1 = flies only in one season (seasonal traveler)
# Score close to 0.25 = flies equally across all seasons (year-round traveler)
season_cols = ['Flights_Fall', 'Flights_Spring', 'Flights_Summer', 'Flights_Winter']
feature_matrix['Season_Concentration'] = (
    feature_matrix[season_cols].max(axis=1) /
    feature_matrix[season_cols].sum(axis=1).replace(0, 1)
)

print("New features added: CLV_log, Flight_consistency_score, Season_Concentration")
print(feature_matrix[['CLV_log', 'Flight_consistency_score', 'Season_Concentration']].describe().round(2))


### Step 13 — Drop Redundant Columns

Removing columns that add noise without meaningful signal.

| Column | Reason for dropping |
|---|---|
| Country | Zero variance — everyone is Canada |
| City | Too granular — Province/Region already captures geography |
| Postal Code | Thousands of unique values — no learnable pattern |
| Enrollment Year | Tenure_Months already captures this |
| Enrollment Month | Tenure_Months already captures this |
| CLV | CLV_log replaces it |
| Province | Region replaces it |


In [ ]:
cols_to_drop = ['Country', 'City', 'Postal Code',
                'Enrollment Year', 'Enrollment Month',
                'CLV', 'Province']

feature_matrix = feature_matrix.drop(
    columns=[c for c in cols_to_drop if c in feature_matrix.columns]
)

print(f"Shape after dropping redundant columns: {feature_matrix.shape}")


### Step 14 — Encoding

**One Hot Encoding** for unordered categories (Gender, Marital Status, Enrollment Type, Region)
**Label Encoding** for ordered categories (Loyalty Card, Education)

**Why the distinction matters for K-Means:**
K-Means uses Euclidean distance. Label encoding unordered categories implies
false order (2 > 1 > 0) which distorts distance calculations.
One hot encoding treats each category as equally different — correct for clustering.


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# One hot encode unordered categoricals
ct = ColumnTransformer(transformers=[
    ('Onehot', OneHotEncoder(drop='first'),
     [feature_matrix.columns.get_loc(c) for c in
      ['Gender', 'Marital Status', 'Enrollment Type', 'Region']])
])

encoded_ar = ct.fit_transform(feature_matrix)
enc_f = ct.get_feature_names_out()

df = feature_matrix.copy()
for i in range(len(enc_f)):
    df[enc_f[i]] = encoded_ar[:, i]

# Drop original categorical columns
df = df.drop(['Gender', 'Marital Status', 'Enrollment Type', 'Region'], axis=1)

# Label encode ordered categoricals
df['loyalty_card_encoded'] = df['Loyalty Card'].map({'Star': 0, 'Nova': 1, 'Aurora': 2})
df['Education_encoded'] = df['Education'].map({
    'High School or Below': 0,
    'College': 1,
    'Bachelor': 2,
    'Master': 3,
    'Doctor': 4
})

# Drop original columns
df = df.drop(['Loyalty Card', 'Education'], axis=1)

print(f"Shape after encoding: {df.shape}")
print(f"Columns: {list(df.columns)}")


### Step 15 — Save Feature Matrix

Saving the final model-ready dataset.
`Loyalty Number` removed — ID column, not a feature.


In [ ]:
df_model = df.drop('Loyalty Number', axis=1)

df_model.to_csv('modelling2.csv', index=False)
print(f"Saved: modelling2.csv")
print(f"Shape: {df_model.shape}")
print(f"Null values: {df_model.isnull().sum().sum()}")
print(f"\nFinal columns ({len(df_model.columns)} total):")
for i, col in enumerate(df_model.columns):
    print(f"  {i+1:2}. {col}")


### Phase 2 — Summary

| Step | Action | Output |
|---|---|---|
| 1 | Load files, remove duplicates | cfa: 391,014 rows, clh: 16,737 rows |
| 2 | Fix negative salaries | 20 values → absolute value |
| 3 | Fix student salary nulls | 4,238 College students → 0 |
| 4 | Build master dataset | 391,014 rows × 25 columns |
| 5 | Build churn label | 2,686 churned (16.0%) |
| 6 | Demographic features | 9 customer-level features |
| 7 | Geographic feature | Province → Region (East/West/North) |
| 8 | Behavioral features | Flights, distance, points, redemption rate |
| 9 | Recency features | Months since last flight, zero activity months |
| 10 | Seasonal features | Flights per season |
| 11 | Combine | 16,737 rows × 29 columns |
| 12 | New features | CLV_log, Flight_consistency_score, Season_Concentration |
| 13 | Drop redundant | Country, City, Postal Code, Enrollment dates, CLV, Province |
| 14 | Encode | One hot (4 cols) + Label encoding (2 cols) |
| 15 | Save | modelling2.csv — model-ready |


---
# Phase 3 — Machine Learning Modelling

## Task A — Churn Prediction Model (XGBoost)

**Goal:** Build a supervised ML model that predicts which customers
are likely to churn based on their behavioral and demographic features.

**Algorithm: XGBoost** — chosen because:
- Handles mixed feature types (numerical + encoded categorical)
- Robust to outliers and skewed distributions
- Provides feature importance via SHAP
- Handles class imbalance via `scale_pos_weight` parameter
- Proven strong performer on tabular classification tasks

**Target variable:** `churned` (0 = active, 1 = churned)
**Class distribution:** 84% not churned, 16% churned → imbalanced dataset


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report,
                              confusion_matrix,
                              roc_auc_score,
                              roc_curve)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load feature matrix
df = pd.read_csv('modelling2.csv')

print(f"Shape: {df.shape}")
print(f"\nChurn distribution:")
print(df['churned'].value_counts())
print(f"\nChurn rate: {df['churned'].mean()*100:.1f}%")


### Step 2 — Define Features and Target

`X` → all columns except `churned`
`y` → `churned` column (target variable)


In [ ]:
X = df.drop(columns=['churned'])
y = df['churned']

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"\nFeature columns ({len(X.columns)}):")
for col in X.columns:
    print(f"  - {col}")


### Step 3 — Train Test Split

**80% training / 20% testing** with stratification.

`stratify=y` ensures both train and test sets maintain the same 16% churn ratio.
Without stratify, the test set could by random chance have very few churned customers
making evaluation unreliable.

`random_state=42` ensures reproducibility — same split every time the code runs.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"\nTrain churn rate : {y_train.mean()*100:.1f}%")
print(f"Test churn rate  : {y_test.mean()*100:.1f}%")


### Step 4 — Handle Class Imbalance

Dataset is imbalanced — 84% not churned, 16% churned.
Without correction, XGBoost will be biased toward predicting 0 (not churned) for every
customer since that is correct 84% of the time — but useless for identifying actual churners.

**Solution:** `scale_pos_weight = count(0) / count(1)`

This tells XGBoost to penalize missing a churned customer more heavily during training.
Effectively balances class weights without oversampling or undersampling.


In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

print(f"Not churned (0) : {neg:,}")
print(f"Churned (1)     : {pos:,}")
print(f"scale_pos_weight: {scale:.2f}")


### Step 5 — Build and Train XGBoost Model

**Key hyperparameters:**

| Parameter | Value | Reason |
|---|---|---|
| n_estimators | 300 | More trees = better learning |
| max_depth | 4 | Shallow trees prevent overfitting |
| learning_rate | 0.05 | Low rate + more trees = better generalisation |
| subsample | 0.8 | Each tree uses 80% of rows — prevents overfitting |
| colsample_bytree | 0.8 | Each tree uses 80% of features — like Random Forest |
| scale_pos_weight | 5.23 | Handles class imbalance |


In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    eval_metric='auc',
    random_state=42,
    verbosity=0
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print(f"Model trained successfully")
print(f"Number of features used: {xgb.n_features_in_}")


### Step 6 — Model Evaluation

**Metrics used:**

- **Accuracy** — overall correct predictions (misleading for imbalanced data)
- **Precision** — of predicted churners, how many actually churned?
- **Recall** — of actual churners, how many did we catch? ← most important
- **F1 Score** — harmonic mean of precision and recall
- **AUC-ROC** — model's ability to distinguish churned vs not churned

**For churn prediction, Recall is most important** — missing a churner who then
leaves is more costly than a false alarm that gets a retention offer.


In [ ]:
y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred,
      target_names=['Not Churned', 'Churned']))
print(f"AUC-ROC Score: {roc_auc_score(y_test, y_prob):.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Churned', 'Churned'],
            yticklabels=['Not Churned', 'Churned'])
plt.title('Confusion Matrix — Churn Prediction')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


**Model Results:**

| Metric | Score | Interpretation |
|---|---|---|
| Accuracy | 97% | 97% of all predictions correct |
| Precision (Churned) | 88% | 88% of flagged churners actually churned |
| Recall (Churned) | 94% | Caught 94% of all actual churners |
| F1 Score | 0.91 | Strong balance of precision and recall |
| AUC-ROC | 0.9883 | Near-perfect discrimination ability |

**Confusion Matrix:**
- True Negatives: 2,741 — correctly identified as not churned
- False Positives: 70 — false alarms
- False Negatives: 31 — missed churners (most costly error)
- True Positives: 506 — correctly identified churners


### Step 7 — ROC Curve

ROC curve plots True Positive Rate (Recall) against False Positive Rate
at different classification thresholds.

AUC = 0.9883 → near-perfect separation between churned and active customers.
The curve hugging the top-left corner confirms this visually.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#534AB7', linewidth=2,
         label=f'XGBoost (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], color='gray',
         linestyle='--', linewidth=1, label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve — Churn Prediction')
plt.legend()
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()


### Step 8 — SHAP Feature Importance

SHAP (SHapley Additive exPlanations) explains which features drive churn predictions.

**Why SHAP over default XGBoost importance:**
- Default importance only shows which features are used most
- SHAP shows actual impact direction and magnitude
- More interpretable for business stakeholders
- Can explain individual customer predictions


In [ ]:
import shap

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_test,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('Top 15 Features — Churn Prediction', fontsize=13)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()


**SHAP Feature Importance — Key Findings:**

| Rank | Feature | Business Meaning |
|---|---|---|
| 1 | Months_Since_Last_Flight | Most powerful signal — dominates by 6x |
| 2 | Tenure_Months | Newer members churn more |
| 3 | Flights_Fall | Fall inactivity = early disengagement warning |
| 4 | Flights_Winter | Winter inactivity confirms seasonal pattern |
| 5 | Total_Flights | Overall low engagement correlates with churn |
| 6 | Zero_Activity_Months | More zero months = higher churn risk |

**Critical insight:** Recency dominates everything. The airline should trigger
a retention alert the moment a customer's last flight crosses 3 months of silence.


### Step 9 — Churn Probability Scores & Risk Tiers

Assigning a probability score (0 to 1) to all 16,737 customers.

**Why probability over binary prediction:**
- Binary loses nuance — 45% risk needs different action than 2% risk
- Marketing can prioritize high-value + high-risk customers first
- Enables cost-effective allocation of retention budget

**Risk tiers:**
- High Risk (>0.5) → immediate action
- Medium Risk (0.2-0.5) → monitoring + light touch
- Low Risk (<0.2) → no intervention needed


In [ ]:
# Churn probabilities for all customers
X_all = df.drop(columns=['churned'])
df['churn_probability'] = xgb.predict_proba(X_all)[:, 1]
df['churn_predicted'] = xgb.predict(X_all)

# Risk tiers
def risk_tier(prob):
    if prob >= 0.5:
        return 'High Risk'
    elif prob >= 0.2:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['risk_tier'] = df['churn_probability'].apply(risk_tier)

print("Risk tier distribution:")
print(df['risk_tier'].value_counts())
print(f"\nAs percentage:")
print(df['risk_tier'].value_counts(normalize=True).mul(100).round(1))
print(f"\nActual churn rate by risk tier:")
print(df.groupby('risk_tier')['churned'].mean().mul(100).round(1))

# Plot
plt.figure(figsize=(8, 5))
tier_counts = df['risk_tier'].value_counts()
colors = ['#E8593C', '#F5A623', '#2ECC71']
bars = plt.bar(tier_counts.index, tier_counts.values,
               color=colors, edgecolor='white')
plt.title('Customer Risk Tier Distribution', fontsize=13)
plt.xlabel('Risk Tier')
plt.ylabel('Number of Customers')
for bar, count in zip(bars, tier_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 50,
             f'{count:,}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('risk_tiers.png', dpi=150)
plt.show()


**Risk Tier Validation:**

| Risk Tier | Customers | % of Total | Actual Churn Rate |
|---|---|---|---|
| High Risk | 2,836 | 16.9% | 92.3% ✓ |
| Medium Risk | 536 | 3.2% | 8.2% ✓ |
| Low Risk | 13,365 | 79.9% | 0.2% ✓ |

**Business impact:** Instead of sending retention campaigns to all 16,737 customers,
the airline can focus on just 3,372 (20.1%) and catch 92%+ of actual churners.
That's a massive efficiency gain in retention spend.


---
## Task B — Customer Segmentation (K-Means)

**Goal:** Group 16,737 customers into meaningful segments
based on behavioral and demographic patterns.

**Algorithm: K-Means** — chosen because:
- Simple and interpretable
- Works well on scaled numerical features
- Produces hard cluster assignments per customer
- Easy to profile and name each segment

**Key question:** "What distinct types of customers does this airline have,
and how do their behaviors and values differ?"

**Important:** `churned` column excluded from clustering features.
Including it would bias clusters toward churn status rather than
genuine behavioral patterns — defeating the purpose of segmentation.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Load modelling dataset
df_cluster = pd.read_csv('modelling2.csv')
df_original = df_cluster.copy()

# Exclude target variable for clustering
cluster_features = df_cluster.drop(columns=['churned'])

# Scale features — K-Means is distance-based
# Without scaling, high-magnitude features like Salary dominate
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_features)

print(f"Clustering features shape: {cluster_features.shape}")
print(f"Features used: {list(cluster_features.columns)}")


### Step 2 — Find Optimal Number of Clusters

Two methods used together to decide the number of clusters:

**Elbow Method:** Plots inertia vs k. The "elbow" where the curve flattens is optimal k.

**Silhouette Score:** Measures how similar a customer is to their own cluster vs other clusters.
Range -1 to +1. Higher = better separated clusters.

Testing k = 2 to 8. Business constraint: segments must be interpretable and actionable.
Too many clusters loses meaning for marketing teams.


In [ ]:
inertias = []
silhouette_scores = []
k_range = range(2, 9)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(cluster_scaled)
    inertias.append(kmeans.inertia_)
    sil = silhouette_score(cluster_scaled, labels)
    silhouette_scores.append(sil)
    print(f"k={k} | Inertia: {kmeans.inertia_:,.0f} | Silhouette: {sil:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertias, 'o-', color='#534AB7', linewidth=2)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')
ax1.grid(True, alpha=0.3)

ax2.plot(k_range, silhouette_scores, 'o-', color='#1D9E75', linewidth=2)
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score')
ax2.grid(True, alpha=0.3)

plt.suptitle('Optimal K Selection', fontsize=13)
plt.tight_layout()
plt.savefig('optimal_k.png', dpi=150)
plt.show()


**K Selection Result: k=3**

| k | Inertia | Silhouette | Decision |
|---|---|---|---|
| 2 | 300,697 | 0.2650 | Too few |
| **3** | **252,769** | **0.2931** | **✓ Selected** |
| 4 | 235,607 | 0.1984 | Silhouette drops sharply |
| 5+ | <222,000 | <0.20 | Diminishing returns |

Both methods agree — k=3 gives the steepest elbow drop AND the highest silhouette score.


### Step 3 — Train Final K-Means with k=3

`n_init=10` runs K-Means 10 times with different starting points and keeps the best result.
This prevents getting stuck in local minima.


In [ ]:
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(cluster_scaled)

df_original['Cluster'] = cluster_labels

print("Cluster distribution:")
print(pd.Series(cluster_labels).value_counts().sort_index())
print(f"\nAs percentage:")
print(pd.Series(cluster_labels).value_counts(
    normalize=True).sort_index().mul(100).round(1))


### Step 4 — Profile Each Cluster

Profiling uses **original unscaled values** — scaled values are meaningless for interpretation.
"Salary = 0.87" means nothing. "Salary = 72,000" is immediately understandable.


In [ ]:
profile_cols = [
    'Salary', 'CLV', 'Tenure_Months',
    'Total_Flights', 'Avg_Flights_Month',
    'Total_Distance', 'Total_Points_Acc',
    'Total_Points_Red', 'Redemption_Rate',
    'Months_Since_Last_Flight', 'Zero_Activity_Months',
    'Flights_Fall', 'Flights_Spring',
    'Flights_Summer', 'Flights_Winter',
    'churned', 'Flight_consistency_score'
]

profile = df_original.groupby('Cluster')[profile_cols].mean().round(2)
print("Cluster profiles — mean values:")
print(profile.T.to_string())


### Step 5 — Name and Visualize Segments

**Three distinct segments identified:**

| Segment | Size | Churn Rate | Avg Tenure | Key Characteristic |
|---|---|---|---|---|
| High-Frequency New Flyers | 835 (5%) | 11.3% | 9 months | Fly most, never redeem, newest members |
| Disengaged At-Risk Members | 4,420 (26.4%) | 53.4% | 24.6 months | Inactive 21/24 months, draining points |
| Loyal Core Members | 11,482 (68.6%) | 2.0% | 45.7 months | Longest tenured, steady, backbone |

**Critical insight:** Loyalty card tier (Star/Nova/Aurora) has almost identical
distribution across all three segments. This proves that tier alone does NOT identify
valuable or at-risk customers. **Behavioral patterns are the true differentiators.**


In [ ]:
segment_names = {
    0: 'High-Frequency New Flyers',
    1: 'Disengaged At-Risk Members',
    2: 'Loyal Core Members'
}
df_original['Segment'] = df_original['Cluster'].map(segment_names)

# 4-panel profile chart
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Customer Segment Profiles', fontsize=14, fontweight='bold')

colors = ['#534AB7', '#E8593C', '#1D9E75']
segments = ['High-Frequency New Flyers',
            'Disengaged At-Risk Members',
            'Loyal Core Members']

# Panel 1 — Avg Flights per Month
ax1 = axes[0, 0]
vals = df_original.groupby('Segment')['Avg_Flights_Month'].mean().reindex(segments)
bars = ax1.bar(range(3), vals.values, color=colors, edgecolor='white')
ax1.set_xticks(range(3))
ax1.set_xticklabels(['High-Freq\nNew', 'Disengaged\nAt-Risk', 'Loyal\nCore'], fontsize=9)
ax1.set_title('Avg Flights per Month')
ax1.set_ylabel('Flights')
for bar, val in zip(bars, vals.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}', ha='center', fontsize=9)

# Panel 2 — Churn Rate
ax2 = axes[0, 1]
vals2 = df_original.groupby('Segment')['churned'].mean().mul(100).reindex(segments)
bars2 = ax2.bar(range(3), vals2.values, color=colors, edgecolor='white')
ax2.set_xticks(range(3))
ax2.set_xticklabels(['High-Freq\nNew', 'Disengaged\nAt-Risk', 'Loyal\nCore'], fontsize=9)
ax2.set_title('Churn Rate (%)')
ax2.set_ylabel('Churn %')
for bar, val in zip(bars2, vals2.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontsize=9)

# Panel 3 — Tenure Months
ax3 = axes[1, 0]
vals3 = df_original.groupby('Segment')['Tenure_Months'].mean().reindex(segments)
bars3 = ax3.bar(range(3), vals3.values, color=colors, edgecolor='white')
ax3.set_xticks(range(3))
ax3.set_xticklabels(['High-Freq\nNew', 'Disengaged\nAt-Risk', 'Loyal\nCore'], fontsize=9)
ax3.set_title('Average Tenure (Months)')
ax3.set_ylabel('Months')
for bar, val in zip(bars3, vals3.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}', ha='center', fontsize=9)

# Panel 4 — Redemption Rate
ax4 = axes[1, 1]
vals4 = df_original.groupby('Segment')['Redemption_Rate'].mean().reindex(segments)
bars4 = ax4.bar(range(3), vals4.values, color=colors, edgecolor='white')
ax4.set_xticks(range(3))
ax4.set_xticklabels(['High-Freq\nNew', 'Disengaged\nAt-Risk', 'Loyal\nCore'], fontsize=9)
ax4.set_title('Redemption Rate')
ax4.set_ylabel('Rate (0-1)')
for bar, val in zip(bars4, vals4.values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('segment_profiles.png', dpi=150)
plt.show()


### Step 6 — Segment vs Risk Tier Cross Table

Combining churn risk scores with segment labels to enable targeted action.


In [ ]:
# Add churn probability from Task A
# (assumes df_original already has churn_probability from Task A)
# If running independently, reload from modelling2.csv and rerun XGBoost

def risk_tier(prob):
    if prob >= 0.5:
        return 'High Risk'
    elif prob >= 0.2:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_original['risk_tier'] = df_original['churn_probability'].apply(risk_tier)

print("Segment vs Risk Tier cross table:")
print(pd.crosstab(df_original['Segment'],
                  df_original['risk_tier'],
                  margins=True))


**Cross Table Interpretation:**

| Segment | High Risk | Key Insight |
|---|---|---|
| Disengaged At-Risk | 54.5% | Over half already high risk — immediate action |
| High-Freq New Flyers | 19.0% | Frequent flying ≠ loyalty — early intervention needed |
| Loyal Core | 2.3% | Very stable — monitor only |

**The most alarming finding:** 19% of High-Frequency New Flyers are already high risk
despite being the most active flyers. Frequent flying without program engagement
(redemption rate = 0.00) means they are transactional, not loyal.


### Step 7 — Save Final Segmented Dataset


In [ ]:
df_original.to_csv('final_customer_segments.csv', index=False)

print("Final dataset saved: final_customer_segments.csv")
print(f"\nShape: {df_original.shape}")
print(f"\nSegment distribution:")
print(df_original['Segment'].value_counts())
print(f"\nRisk tier distribution:")
print(df_original['risk_tier'].value_counts())
print(f"\nColumns: {list(df_original.columns)}")


---
# Phase 3 — Complete Summary

## Task A — Churn Prediction Model (XGBoost)

**Model:** XGBoost Classifier
**Features:** 33 behavioral and demographic features
**Target:** churned (0 = active, 1 = churned)

**Performance:**
- Accuracy      : 97%
- Precision     : 88% (churned class)
- Recall        : 94% (churned class)
- F1 Score      : 0.91
- AUC-ROC       : 0.9883

**Key findings:**
- `Months_Since_Last_Flight` is dominant predictor — 6x more important than any other feature
- `Tenure_Months` second most important — newer members churn more
- Seasonal inactivity (Fall, Winter) signals early disengagement
- Model catches 94% of all churned customers — only 31 missed out of 537 in test set

**Risk tier output:**
- High Risk   : 2,836 customers (16.9%) — 92.3% actually churned ✓
- Medium Risk :   536 customers  (3.2%) — 8.2% actually churned ✓
- Low Risk    : 13,365 customers (79.9%) — 0.2% actually churned ✓

---

## Task B — Customer Segmentation (K-Means)

**Algorithm:** K-Means, k=3
**Silhouette Score:** 0.2931
**Features:** 32 scaled behavioral and demographic features

**Three segments identified:**

| Segment | Size | Churn Rate | Avg Tenure | Key Signal |
|---|---|---|---|---|
| High-Frequency New Flyers | 835 (5.0%) | 11.3% | 9 months | Fly most, never redeem |
| Disengaged At-Risk Members | 4,420 (26.4%) | 53.4% | 24.6 months | Inactive, draining points |
| Loyal Core Members | 11,482 (68.6%) | 2.0% | 45.7 months | Steady, longest tenured |

**Key insights:**
1. Loyalty card tier does NOT differentiate segments — behavioral patterns do
2. Redemption rate 0.97 for disengaged members is a powerful exit signal
3. 19% of High-Frequency New Flyers are already high risk — frequent flying ≠ loyalty
4. Flight consistency score clearly separates consistent flyers from bursty ones

**Segment vs Risk Tier:**
- Disengaged At-Risk : 54.5% high risk → immediate action
- High-Freq New      : 19.0% high risk → early intervention
- Loyal Core         : 2.3%  high risk → monitor only

---

## Output Files

| File | Description |
|---|---|
| `modelling2.csv` | 16,737 rows × 33 features — model-ready |
| `final_customer_segments.csv` | Complete master file with segments + risk tiers |
| `confusion_matrix.png` | Model evaluation visual |
| `roc_curve.png` | AUC-ROC curve |
| `shap_importance.png` | Feature importance by SHAP |
| `optimal_k.png` | K selection — elbow + silhouette |
| `segment_profiles.png` | 4-panel segment comparison |
